# 🧠 GEN3IA — Entraînement GPU : photoréalisme + voix humaine + vidéo**Objectif :** donner à Gen3 la capacité de générer des **images photoréalistes**, des **voix humaines** et des **vidéos réalistes**, avec des modèles dont l'inférence tournera ensuite **dans le navigateur des utilisateurs via WebGPU** (zéro serveur d'IA externe, philosophie Gen3ia conservée).**Comment exécuter :**1. Menu `Exécution` → `Modifier le type d'exécution` → **GPU T4** (ou L4)2. `Exécution` → **Tout exécuter** (≈ 25–40 min)3. Les artefacts sont publiés dans le repo Hugging Face `Kings237/gen3ia-media-engine` — le navigateur de gen3ia.online y puisera les modèles.⚠️ **Sécurité :** le token HF est fourni par le propriétaire du projet — pensez à le **révoquer** après la session (https://huggingface.co/settings/tokens).

In [ ]:
# ═══════════════ CELLULE 1 — Dépendances ═══════════════%pip -q install -U diffusers transformers peft accelerate safetensors     onnx onnxruntime-gpu huggingface_hub sentencepiece einopsimport os, json, math, time, hashlib, warningsimport torchwarnings.filterwarnings("ignore")print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ AUCUN — active le GPU (Runtime → Modifier le type)")assert torch.cuda.is_available(), "Ce notebook requiert un GPU Colab.

In [ ]:
# ═══════════════ CELLULE 2 — Connexion Hugging Face ═══════════════# Token fourni par le propriétaire du projet Gen3ia (révocable à tout moment).HF_TOKEN = os.environ.get("HF_TOKEN", "hf_VOTRE_TOKEN_ICI")from huggingface_hub import login, HfApi, create_repologin(HF_TOKEN)api = HfApi()WHO = api.whoami()["name"]HF_REPO = f"{WHO}/gen3ia-media-engine"create_repo(HF_REPO, repo_type="model", private=False, exist_ok=True)print(f"✅ Connecté : {WHO} — artefacts publiés dans {HF_REPO} (public, lisible par le navigateur)

In [ ]:
# ═══════════════ CELLULE 3 — Datasets Gen3ia ═══════════════# Les 4 datasets (abréviations, images, voix, vidéo) sont déjà publiés dans# le repo gen3ia-media-engine. On les télécharge et on en extrait les briefs# photoréalistes qui STYLISENT le fine-tune (prompts enrichis Gen3ia).from huggingface_hub import hf_hub_downloadimport json as _jsonDATASETS = {}for name in ["abbreviations", "images", "voice", "video"]:    p = hf_hub_download(HF_REPO, f"datasets/{name}/train.jsonl", repo_type="model")    rows = [ _json.loads(l) for l in open(p, encoding="utf-8") if l.strip() ]    DATASETS[name] = rows    print(f"  {name}: {len(rows):,} lignes")# Prompts photoréalistes Gen3ia (colonne INPUT du dataset images)photoreal_prompts = [r["input"] for r in DATASETS["images"]                     if r.get("task_type") == "prompt_to_photoreal_brief"]print(f"✅ {len(photoreal_prompts)} briefs photoréalistes Gen3ia chargés

In [ ]:
# ═══════════════ CELLULE 4 — Photos réelles (COCO val2017) ═══════════════# Le photoréalisme s'apprend sur de VRAIES photos avec de VRAIES légendes.# COCO val2017 : 5 000 photos réelles + 5 légendes chacune (CC-BY).!mkdir -p /content/coco%cd /content/coco!wget -q http://images.cocodataset.org/zips/val2017.zip -O val2017.zip!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O ann.zip!unzip -qo val2017.zip && unzip -qo ann.zip%cd /contentimport json as _jann = _j.load(open("/content/coco/annotations/captions_val2017.json"))caps_by_img = {}for c in ann["annotations"]:    caps_by_img.setdefault(c["image_id"], []).append(c["caption"].strip())img_by_id = {im["id"]: im["file_name"] for im in ann["images"]}coco_pairs = []for img_id, file_name in img_by_id.items():    if img_id in caps_by_img and len(caps_by_img[img_id]) >= 1:        coco_pairs.append({            "image": f"/content/coco/val2017/{file_name}",            "prompt": caps_by_img[img_id][0],        })print(f"✅ {len(coco_pairs)} paires (photo réelle, légende) prêtes pour le fine-tune

In [ ]:
# ═══════════════ CELLULE 5 — Fine-tune LoRA PHOTORÉALISME (SD 2.1-base) ═══════════════# Choix du modèle : stable-diffusion-2-1-base (licence CreativeML Open RAIL-XXM,# usage commercial autorisé, UNet 865 M — taille adaptée à l'inférence WebGPU).# LoRA (rang 16) : on n'entraîne que ~0.6 % des poids → rapide et léger.import random, globfrom torch.utils.data import Dataset, DataLoaderfrom PIL import Imagefrom diffusers import StableDiffusionPipeline, DDPMSchedulerimport torch.nn.functional as FMODEL_ID = "stabilityai/stable-diffusion-2-1-base"RESOLUTION = 512BATCH = 4STEPS = 600          # ≈ 25 min sur T4 — augmentable à 1500 pour + de fidélitéLR = 1e-4pipe = StableDiffusionPipeline.from_pretrained(    MODEL_ID, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)pipe = pipe.to("cuda")pipe.set_progress_bar_config(disable=True)unet = pipe.unetvae = pipe.vaetext_encoder = pipe.text_encodertokenizer = pipe.tokenizer# --- LoRA sur l'UNetlora_cfg = LoraConfig(    r=16, lora_alpha=32, lora_dropout=0.0, bias="none",    target_modules=["to_q", "to_k", "to_v", "to_out.0"],)from peft import get_peft_modelunet.enable_gradient_checkpointing()unet_lora = get_peft_model(unet, lora_cfg)unet_lora.train()# --- Dataset : photos COCO (70 %) + briefs Gen3ia repris en prompts (30 %)class PhotoRealDS(Dataset):    def __init__(self, pairs, prompts, size=RESOLUTION):        self.pairs = pairs        self.prompts = prompts        self.size = size    def __len__(self):        return len(self.pairs) + len(self.prompts)    def __getitem__(self, i):        if i < len(self.pairs):            p = self.pairs[i]            img = Image.open(p["image"]).convert("RGB")            prompt = p["prompt"]        else:            img = Image.open(random.choice(self.pairs)["image"]).convert("RGB")            prompt = random.choice(self.prompts)  # style Gen3ia sur photo réelle        img = img.resize((self.size, self.size), Image.LANCZOS)        arr = (torch.from_numpy(__import__("numpy").array(img)).float() / 127.5 - 1.0).permute(2, 0, 1)        return {"pixel": arr, "prompt": prompt}ds = PhotoRealDS(coco_pairs, photoreal_prompts)dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, persistent_workers=True)optimizer = torch.optim.AdamW([p for p in unet_lora.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2)noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)STEP_HISTORY = []step = 0t0 = time.time()while step < STEPS:    for batch in dl:        if step >= STEPS: break        with torch.no_grad():            latents = vae.encode(batch["pixel"].half().cuda()).latent_dist.sample()            latents = latents * vae.config.scaling_factor            ids = tokenizer(batch["prompt"], padding="max_length", max_length=tokenizer.model_max_length,                            truncation=True, return_tensors="pt").input_ids            emb = text_encoder(ids.cuda(), attention_mask=None)[0].half()        noise = torch.randn_like(latents)        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device="cuda").long()        noisy = noise_scheduler.add_noise(latents, noise, timesteps)        pred = unet_lora(noisy, timesteps, emb).sample        loss = F.mse_loss(pred.float(), noise.float(), reduction="mean")        optimizer.zero_grad(set_to_none=True)        loss.backward()        torch.nn.utils.clip_grad_norm_([p for p in unet_lora.parameters() if p.requires_grad], 1.0)        optimizer.step()        step += 1        if step % 50 == 0 or step == 1:            STEP_HISTORY.append({"step": step, "loss": round(loss.item(), 4)})            print(f"  step {step}/{STEPS} — loss {loss.item():.4f} — {time.time()-t0:.0f}s")print("✅ Fine-tune terminé — perte finale :", STEP_HISTORY[-1] if STEP_HISTORY else "n/a

In [ ]:
# ═══════════════ CELLULE 6 — Fusion LoRA + test de rendu ═══════════════unet = unet_lora.merge_and_unload()pipe.unet = unetdel unet_loratorch.cuda.empty_cache()gen = torch.Generator("cuda").manual_seed(42)test_prompt = "a photo of " + random.choice(photoreal_prompts).replace("Une photo de ", "").replace("A photo of ", "")img = pipe(test_prompt, num_inference_steps=25, guidance_scale=4.5, height=512, width=512,           generator=gen).images[0]os.makedirs("/content/gen3ia_artifacts", exist_ok=True)img.save("/content/gen3ia_artifacts/test_photoreal.png")print("🖼️ Test :", test_prompt)img

In [ ]:
# ═══════════════ CELLULE 7 — Export ONNX fp16 (inférence WebGPU navigateur) ═══════════════# Le navigateur des utilisateurs exécutera text_encoder + unet + vae_decoder# via onnxruntime-web (backend WebGPU). On exporte les 3 modules en fp16.!pip -q install "diffusers[onnx]" optimum onnx onnxruntime-gpuimport torch as _tfrom diffusers import UNet2DConditionModel, AutoencoderKL, CLIPTextModelEXPORT = "/content/gen3ia_artifacts/sd-photoreal"os.makedirs(EXPORT, exist_ok=True)class UnetWrap(_t.nn.Module):    def __init__(self, u): super().__init__(); self.u = u    def forward(self, sample, timestep, encoder_hidden_states):        return self.u(sample, timestep, encoder_hidden_states, return_dict=False)[0]unet_cpu = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=_t.float16)unet_cpu.load_state_dict(unet.state_dict(), strict=False)unet_cpu = UnetWrap(unet_cpu.half()).eval()vae_cpu = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=_t.float16).eval()enc_cpu = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=_t.float16).eval()dummy_ids = _t.ones((1, 77), dtype=_t.long)dummy_lat = _t.randn((1, 4, 64, 64), dtype=_t.float16)dummy_ts = _t.tensor([500], dtype=_t.float16)with _t.no_grad():    _t.onnx.export(enc_cpu, (dummy_ids,), f"{EXPORT}/text_encoder.onnx",        input_names=["input_ids"], output_names=["last_hidden_state", "pooler_output"],        dynamic_axes={"input_ids": {0: "batch"}, "last_hidden_state": {0: "batch"}}, opset_version=17)    _t.onnx.export(unet_cpu, (dummy_lat, dummy_ts, _t.randn((1, 77, 1024), dtype=_t.float16)),        f"{EXPORT}/unet.onnx",        input_names=["sample", "timestep", "encoder_hidden_states"], output_names=["out_sample"],        dynamic_axes={"sample": {0: "batch"}, "encoder_hidden_states": {0: "batch"}, "out_sample": {0: "batch"}},        opset_version=17)    _t.onnx.export(vae_cpu.decode, (dummy_lat,), f"{EXPORT}/vae_decoder.onnx",        input_names=["latent"], output_names=["image"],        dynamic_axes={"latent": {0: "batch"}, "image": {0: "batch"}}, opset_version=17)# Tokenizer CLIP (vocab + merges) + config du scheduler pour le clienttokenizer.save_pretrained(EXPORT)import json as _j_j.dump({"scheduler": "ddim", "steps_browser": 4, "guidance": 1.0,         "note": "SD2.1-base + LoRA photoréalisme Gen3ia — exécution WebGPU ort-web"},        open(f"{EXPORT}/client_config.json", "w"), indent=2)print("✅ ONNX exportés :", sorted(os.listdir(EXPORT))

In [ ]:
# ═══════════════ CELLULE 8 — Voix humaine (Kokoro-82M, Apache-2.0) ═══════════════# Kokoro-82M : TTS neuronal Compact — voix FR naturelle `ff_siwis`, licence# Apache-2.0 (usage commercial OK), ~82 M de paramètres → tourne dans le# navigateur (transformers.js / ort-web, WebGPU).!mkdir -p /content/gen3ia_artifacts/kokoro/voicesK = "onnx-community/Kokoro-82M-v1.0-ONNX"for f in ["config.json", "model.onnx"]:    p = hf_hub_download(K, f); os.system(f"cp '{p}' /content/gen3ia_artifacts/kokoro/")p = hf_hub_download(K, "voices.bin")os.system(f"cp '{p}' /content/gen3ia_artifacts/kokoro/voices/")try:    vj = hf_hub_download(K, "voices.json"); os.system(f"cp '{vj}' /content/gen3ia_artifacts/kokoro/voices.json")except Exception as e:    print("voices.json absent :", e)print("✅ Kokoro prêt :", sorted(os.listdir("/content/gen3ia_artifacts/kokoro")))# NOTE ENTRAÎNEMENT : le style vocal Gen3ia (datasets/voice) module les# paramètres côté client (débit, prosodie, pauses) — le modèle de base# couvre la voix humaine naturelle ; un fine-tune Piper FR reste possible# plus tard (cellule optionnelle en fin de notebook).

In [ ]:
# ═══════════════ CELLULE 9 — Vidéo : keyframes → MP4 (test réel) ═══════════════# Stratégie vidéo Gen3 : les keyframes photoréalistes sont générées par le# UNet fin-tuné (cellule 6), l'assemblage MP4 (fondus, transitions) est fait# par ffmpeg — sur VPS côté serveur, ffmpeg.wasm côté navigateur.!apt -qq install -y ffmpeg > /dev/null 2>&1scenes = [random.choice(photoreal_prompts) for _ in range(3)]frames = []for i, sp in enumerate(scenes):    prompt = sp if sp.lower().startswith(("a ", "an ")) else "a photo of " + sp.replace("Une photo de ", "")    im = pipe(prompt, num_inference_steps=20, guidance_scale=4.0, height=512, width=512,              generator=torch.Generator("cuda").manual_seed(100 + i)).images[0]    frames.append(f"/content/kf_{i}.png"); im.save(frames[-1])    print(f"  keyframe {i+1}/3 : {prompt[:80]}")os.system(    "ffmpeg -y -loop 1 -t 1.4 -i /content/kf_0.png -loop 1 -t 1.4 -i /content/kf_1.png "    "-loop 1 -t 1.4 -i /content/kf_2.png -filter_complex "    "[0][1]xfade=transition=fade:duration=0.5:offset=0.9[v01];[v01][2]xfade=transition=fade:duration=0.5:offset=2.3[vf];"    "[vf]scale=640:-2,fps=24,format=yuv420p[vout] -map [vout] -c:v libx264 -preset veryfast "    "/content/gen3ia_artifacts/test_keyframes.mp4")print("✅ MP4 de test :", os.path.getsize("/content/gen3ia_artifacts/test_keyframes.mp4"), "octets

In [ ]:
# ═══════════════ CELLULE 10 — Publication Hugging Face ═══════════════import json as _j, hashlib as _hdef sha(p):    h = _h.sha256()    with open(p, "rb") as f:        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)    return h.hexdigest()[:16]MANIFEST = {    "name": "gen3ia-media-engine",    "version": 1,    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),    "training": {        "photorealism": {"base": MODEL_ID, "method": "LoRA r16 (fusé)", "photos": len(coco_pairs),                         "prompts_gen3ia": len(photoreal_prompts), "steps": STEPS,                         "loss_history": STEP_HISTORY},        "voice": {"model": "Kokoro-82M", "license": "Apache-2.0", "voice_fr": "ff_siwis"},        "video": {"method": "keyframes photoréalistes + ffmpeg xfade", "fps": 24},    },    "browser_inference": {        "runtime": "onnxruntime-web (WebGPU) + kokoro-js",        "sd": {"dir": "sd-photoreal", "dtype": "fp16", "steps": 4, "size_class": "SD2.1-base (865M UNet)"},        "kokoro": {"dir": "kokoro", "voice_fr": "ff_siwis", "dtype": "fp32"},    },    "datasets": {k: len(v) for k, v in DATASETS.items()},}_j.dump(MANIFEST, open("/content/gen3ia_artifacts/MANIFEST.json", "w"), indent=2, ensure_ascii=False)api.upload_folder(    repo_id=HF_REPO, repo_type="model", folder_path="/content/gen3ia_artifacts",    commit_message="Gen3ia media engine v1 — LoRA photoréalisme (COCO) + Kokoro voix + keyframes vidéo",)print("🚀 Publié : https://huggingface.co/" + HF_REPO)print(json.dumps({k: MANIFEST[k] for k in ["version", "datasets"]}, indent=2, ensure_ascii=False))

## ✅ Après le notebook — dans gen3ia.online1. `lib/neural/webgpu/*` charge `MANIFEST.json` depuis ce repo et télécharge les ONNX (cache navigateur).2. Demande « génère une image photoréaliste de … » dans le chat → génération sur **le GPU de l'utilisateur** (WebGPU), repli automatique sur le moteur neuronal abstrait si WebGPU est absent.3. « dis / lis à voix haute » → **Kokoro ff_siwis** (voix française humaine) dans le navigateur.4. « fais une vidéo de … » → keyframes WebGPU + assemblage ffmpeg.wasm.**Optionnel — fine-tune voix Piper (FR)** : cellule avancée, dataset CSS10-FR :```python# !git clone https://github.com/rhasspy/piper /content/piper-src# voir piper/src/python — nécessite ~2 h GPU et un corpus WAV+transcript```